# Practice guide 04_03: functionality-preserving evasion and the defence

**Goal.** Attack the PE-header detector of guide 04_01 yourself, three ways: the malleable rewrite, benign
mimicry, and a greedy black-box query attack; measure evasion rates; then implement the adversarial
retraining defence and test whether it generalises to an attack it did not see. Every `TODO` cell must be
completed; the `check` cells tell you whether you got it right.

Read `guide_04_03.pdf` first. Time: about 2 hours. Cells marked **(given)** are complete.

In [ ]:
import sys
from pathlib import Path

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score
from sklearn.model_selection import train_test_split

LIB = next(p for p in (Path("../../notebooks/03-malware"), Path("../../../notebooks/03-malware")) if p.exists())
sys.path.insert(0, str(LIB))  # student / solution folder
import malwlib as mw

## Part A. Setup (given)

The detector of guide 04_01 (recall ≈ .997) and the median benign profile. `mw.MALLEABLE` lists the header
fields an attacker may set freely — rewriting them costs nothing at runtime.

In [ ]:
X, y, cols = mw.load_pe()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
rf = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(Xtr, ytr)
mal = np.where(yte == 1)[0]
med = np.median(Xtr[ytr == 0], 0)
mal_fields = [cols.index(c) for c in mw.MALLEABLE if c in cols]
print(f"baseline malicious recall {recall_score(yte, rf.predict(Xte), pos_label=1):.4f}")
print(f"malleable fields present: {len(mal_fields)} of {len(mw.MALLEABLE)}")


def evasion(model, Xa, mask=mal, thr=0.5):
    """Share of malicious test files scored below thr after the attack."""
    return mw.evasion_rate(model, Xa[mask], np.ones(mask.sum()), thr=thr)

**Question A1.** Why does an attacker prefer functionality-preserving manipulations over, say, appending
bytes that lower the score? State the two constraints every "good" evasion attack must satisfy.

**Answers A1.** The payload must still run and must still do its job: an attack that breaks execution or
changes behaviour turns a working trojan into a crashed sample — worthless (and visible). The two constraints
are (1) **functionality preservation**: the modified program must behave identically on its targets, and
(2) **stealth effectiveness**: the model's score must drop below the alarm threshold. Malleable metadata
satisfies (1) by construction; the whole game is (2).

## Part B. Attack A — malleable rewrite

Set every malleable field of every malicious test file to the benign median. Implement it with `mw.median_rewrite`
and measure the evasion rate.

In [ ]:
XA = mw.median_rewrite(Xte, Xte, mal_fields, med)
evA = evasion(rf, XA)
print(f"attack A evasion {evA:.3f}")

**Question B1.** Attack A reaches evasion 1.000. What does a perfect evasion rate mean operationally — and
which single number of the *undefended* detector would you quote to argue that the benchmark, not the method,
is what failed?

**Answers B1.** Every malicious test file is scored benign: the scanner as deployed provides zero protection
against an attacker who reads this pipeline. The number to quote is the baseline recall 0.9975 — the detector
is near-perfect on the *distribution it was trained on*; the failure is the benchmark's assumption that
header metadata is adversarial-invariant, which the malleable-field list falsifies with one line of code.

## Part C. Attack B — benign mimicry

Copy a random benign profile onto each malicious file and keep only `k` of its fields from the payload.
Sweep k = 40, 20, 10, 5 and record the evasion rates.

In [ ]:
rng = np.random.default_rng(0)
evs = {}
evs = {}
for k in [40, 20, 10, 5]:
    XB = mw.benign_template_attack(Xte[mal], Xtr[ytr == 0], keep=k, rng=rng, cols=X.shape[1])
    XBf = Xte.copy()
    XBf[mal] = XB
    evs[k] = evasion(rf, XBf)
    print(f"keep {k:2d} fields -> evasion {evs[k]:.3f}")

**Question C1.** Evasion *rises* as k falls. Explain the trade-off from the attacker's seat: what does small
k buy, what does it risk, and why is the risk invisible in this experiment?

**Answers C1.** Small k means almost the whole benign profile is copied, so the file looks maximally benign
to the detector — evasion approaches 1. The risk: if a copied field is *not* actually malleable (sizes,
entry point, versions must match the real build), the program may fail to load or behave differently —
functionality preservation is only statistical, not guaranteed. The risk is invisible here because we attack
feature vectors, not real PE files: no experiment ever runs the modified binary. Any mimicry evaluation that
stops at the model score overstates the attack.

## Part D. Attack C — greedy black-box

The realistic attack: the attacker only *queries* the detector. Try each malleable field at its benign
median, apply the single rewrite that lowers the score the most, repeat up to 30 steps. Implement the loop
for the first 60 malicious test files (mw.greedy_evasion is the reference; write yours and check it agrees).

In [ ]:
def my_greedy(model, x, candidates, med, max_steps=30):
    x = x.copy()
    for _ in range(max_steps):
        p = model.predict_proba(x[None].astype(np.float64))[0, 1]
        if p < 0.5:
            break
        cand = np.asarray(candidates)
        Xc = np.tile(x, (len(cand), 1))
        Xc[np.arange(len(cand)), cand] = med[cand]
        pv = model.predict_proba(Xc.astype(np.float64))[:, 1]  # one batched query per step
        k = int(np.argmin(pv))
        if pv[k] >= p:
            break
        x[cand[k]] = med[cand[k]]
    return x

In [ ]:
sub = mal[:60]
XC = Xte.copy()
for i in sub:
    XC[i] = my_greedy(rf, Xte[i], mal_fields, med)
evC = evasion(rf, XC, mask=sub)
ref = np.array([mw.greedy_evasion(rf, x, mal_fields, med) for x in Xte[sub]])
assert np.allclose(XC[sub], ref)
print(f"attack C evasion {evC:.3f} (greedy implementation matches the reference)")

**Question D1.** Attack C reaches the same evasion as A with no gradients and no model access — only query access.
Estimate the query budget: how many `predict_proba` calls does one file cost, at most, and what does that
suggest about rate-limiting scanner APIs?

**Answers D1.** Each step scores up to |candidates| + 1 = 23 models, over at most 30 steps: about 690
queries per file, in practice far fewer because most files drop below 0.5 in a handful of steps. A scanner
API that rate-limits (say 100 queries per client) makes the greedy attack 7x more expensive — black-box
evasion is an economics game, and query metering is a real defence layer (though proxies defeat it).

## Part E. The defence: adversarial retraining

Retrain the forest with attack-A-rewritten *training* malware added to the training set. Then evaluate the
defended model on all three attacks.

In [ ]:
rf_def = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1)
Xadv = {}
Xadv_A = mw.median_rewrite(Xtr, Xtr, mal_fields, med)
Xaug = np.vstack([Xtr, Xadv_A[ytr == 1]])
yaug = np.concatenate([ytr, np.ones((ytr == 1).sum(), int)])
rf_def.fit(Xaug, yaug)
print(f"defended baseline recall {recall_score(yte, rf_def.predict(Xte), pos_label=1):.4f}")
print(f"defended attack A evasion {evasion(rf_def, XA):.3f}")
XB10 = Xte.copy()
XB10[mal] = mw.benign_template_attack(Xte[mal], Xtr[ytr == 0], keep=10, rng=rng, cols=X.shape[1])
print(f"defended attack B (k=10) evasion {evasion(rf_def, XB10):.3f}")
XCs = Xte.copy()
for i in sub:
    XCs[i] = mw.greedy_evasion(rf_def, Xte[i], mal_fields, med)
print(f"defended attack C evasion {evasion(rf_def, XCs, mask=sub):.3f}")

**Question E1.** The defence kills attack A and the greedy attack C (0.02) but mimicry B still evades (0.94).
Why does retraining only patch the region it saw, and what would an *adaptive* attacker do next? Name the arms-race step that ends
this loop for a defender with finite resources.

**Answers E1.** Adversarial retraining adds the A-manifold to the training distribution; the forest redraws
its boundary around the median-rewrite region; the greedy path ends in that same region, so it is closed too
(0.02), but mimicry lands elsewhere and survives (0.94).
The adaptive attacker re-runs the greedy search against the defended model and finds new malleable combos
(or moves to mimicry, which needs no list). The loop ends where the defender stops trusting mutable metadata
altogether — features from execution (04_02) or provenance — or pays for signed/benign-listed channels;
otherwise the attacker's iteration is cheaper than the defender's.

## Part F. The evasion summary table

Fill the table: evasion of the undefended model (A, B k=10, C) and of the defended model (same three).

In [ ]:
rows = {}
rows = {
    ("undefended", "A"): evA,
    ("undefended", "B"): evs[10],
    ("undefended", "C"): evC,
    ("defended", "A"): evasion(rf_def, XA),
    ("defended", "B"): evasion(rf_def, XB10),
    ("defended", "C"): evasion(rf_def, XCs, mask=sub),
}
for (m, a), v in rows.items():
    print(f"{m:10s} {a}  evasion {v:.2f}")

**Question F1.** Which attack is the most robust across both models, and why does that make mimicry the
canonical threat for ML-based malware detectors?

**Answers F1.** Mimicry: it does not depend on which fields the model weights — it replaces the sample with
something drawn from the benign bulk of the training distribution, so any model trained on that distribution
scores it low. Retraining cannot remove the benign region itself. That generality (works white-box or
black-box, survives retraining) makes mimicry the canonical threat, and it is why the field is moving to
execution-based features where "looking benign" is harder to fake.

# Reference solution

The solved notebook, with the answers to the questions and the numbers of a full run, is `solution/guide_04_03.ipynb`.